# Reflecting on Qwen3.5 answers using SMT (submission)

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
from pathlib import Path

from unsloth import FastLanguageModel

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    DATA_DIR,
    PROMPT_REWRITE,
    PROMPTS,
    REFLECTION_TEMPERATURE,
    REFLECTION_TOP_P,
    REFLECTION_TOP_K,
    REFLECTION_MIN_P,
    REFLECTION_MAX_NEW_TOKENS,
    REFLECTION_MAX_SEQUENCE_LENGTH,
    REFLECTION_REPETITION_PENALTY,
)
from staged_qwen3_5_scivqa.smt.reflection import reflect_answers

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-9B"
DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

# Input Files
INITIAL_STATE_PATH = DATA_DIR / f"submission_finetuning_{CATEGORY}_state.json"
SMT_FILE = DATA_DIR / f"smt_{CATEGORY}_state.json"

# Output Files
REFLECTION_STATE_PATH = DATA_DIR / f"submission_reflection_{CATEGORY}_state.json"
FINAL_SUBMISSION_PATH = DATA_DIR / f"submission_final_{CATEGORY}.json"

<a name="Submission"></a>
### Loading the Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    max_seq_length=REFLECTION_MAX_SEQUENCE_LENGTH,
    dtype=None,
)
FastLanguageModel.for_inference(model)

### Running Reflection

In [ ]:
with open(SMT_FILE) as f:
    smt_data = json.load(f)

with open(INITIAL_STATE_PATH) as f:
    initial_state = json.load(f)

In [ ]:
reflected_state = reflect_answers(
    model=model,
    tokenizer=tokenizer,
    initial_state=initial_state,
    smt_data=smt_data,
    reflection_state_path=REFLECTION_STATE_PATH,
    final_submission_path=FINAL_SUBMISSION_PATH,
)

print(f"Final submission saved to {FINAL_SUBMISSION_PATH}.")